In [0]:
from delta.tables import DeltaTable

In [0]:
source_fact_table = "silver.silver_bookings"
target_fact_table = "gold.fact_bookings"
cdc_column = "modified_date"


In [0]:
# This configuration drives the dynamic query generation.
dimensions_config = [
    {"dim_table": "gold.dim_silver_passengers", "lookup_key": "passenger_id", "surrogate_key": "dim_silver_passengers_key"},
    {"dim_table": "gold.dim_silver_flights", "lookup_key": "flight_id", "surrogate_key": "dim_silver_flights_key"},
    {"dim_table": "gold.dim_silver_airports", "lookup_key": "airport_id", "surrogate_key": "dim_silver_airports_key"}
]

In [0]:
# Dynamic query generation
query = "SELECT\n"
for i, dim in enumerate(dimensions_config):
    query += f"  d{i}.{dim['surrogate_key']},\n"
query += "  f.amount,\n  f.booking_date\n"
query += f"FROM {source_fact_table} f\n"
for i, dim in enumerate(dimensions_config):
    query += f"LEFT JOIN {dim['dim_table']} d{i} ON f.{dim['lookup_key']} = d{i}.{dim['lookup_key']}\n"

print("--- Generated SQL Query ---\n" + query)


--- Generated SQL Query ---
SELECT
  d0.dim_silver_passengers_key,
  d1.dim_silver_flights_key,
  d2.dim_silver_airports_key,
  f.amount,
  f.booking_date
FROM silver.silver_bookings f
LEFT JOIN gold.dim_silver_passengers d0 ON f.passenger_id = d0.passenger_id
LEFT JOIN gold.dim_silver_flights d1 ON f.flight_id = d1.flight_id
LEFT JOIN gold.dim_silver_airports d2 ON f.airport_id = d2.airport_id



In [0]:
# Execute query and save as fact table
final_fact_df = spark.sql(query)

# For simplicity, we'll do a full overwrite. In production, you would add MERGE logic.
final_fact_df.write.format("delta").mode("overwrite").saveAsTable(target_fact_table)
print(f"Fact table {target_fact_table} created successfully.")


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5690556894193116>, line 2
      1 # Execute query and save as fact table
----> 2 final_fact_df = spark.sql(query)
      4 # For simplicity, we'll do a full overwrite. In production, you would add MERGE logic.
      5 final_fact_df.write.format("delta").mode("overwrite").saveAsTable(target_fact_table)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/session.py:879, in SparkSession.sql(self, sqlQuery, args, **kwargs)
    876         _views.append(SubqueryAlias(df._plan, name))
    878 cmd = SQL(sqlQuery, _args, _named_args, _views)
--> 879 data, properties, ei = self.client.execute_command(cmd.command(self._client))
    880 if "sql_command_result" in properties:
    881     df = DataFrame(CachedRelation(properties["sql_command_result"]), self)

File /databricks/python/lib/python3.12/site-packages/pys